In [ ]:
!apt-get install openjdk-11-jdk-headless -qq
!pip install -q pyspark findspark

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
# No need to download Spark—pip installation includes Spark JARs

import findspark
findspark.init()

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("ColabSpark").getOrCreate()
spark

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================================
# 3.4 Data Integration (Iteration 4, PySpark version)
# =========================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import json, os
import pandas as pd
from pathlib import Path

# -----------------------------
# Spark init
# -----------------------------
spark = SparkSession.builder.appName("Iteration4_DataIntegration").getOrCreate()

# -----------------------------
# Paths
# -----------------------------
OUT_DIR = Path("/content/drive/MyDrive/Infosys722/iteration4_prep_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Input file ---
IN_PATH = OUT_DIR / "final_constructed_table.parquet"
CODEBOOK_PATH = OUT_DIR / "semantic_codebook_full.json"
OUT_SEMANTIC_CSV = OUT_DIR / "final_integrated_table_semantic.csv"
DICT_CSV = OUT_DIR / "semantic_dictionary.csv"
SNAPSHOT = OUT_DIR / "semantic_codebook_snapshot.json"

assert IN_PATH.exists(), f"❌ Input not found: {IN_PATH}"

# -----------------------------
# 1. Load data
# -----------------------------
# --- Read from parquet instead of CSV ---
df = spark.read.parquet(str(IN_PATH))
print(f"✅ Loaded constructed table: {df.count():,} rows × {len(df.columns)} columns")

# -----------------------------
# 2. Build semantic codebook (same as Iteration 3)
# -----------------------------
CODEBOOK = {
    # --- Demographics ---
    "SEX": {"1": "Male", "2": "Female"},
    "REGION": {"1": "Northeast", "2": "Midwest", "3": "South", "4": "West"},
    "R_MARITL": {
        "0": "Under 14 years",
        "1": "Married - spouse in household",
        "2": "Married - spouse not in household",
        "3": "Married - spouse in household unknown",
        "4": "Widowed",
        "5": "Divorced",
        "6": "Separated",
        "7": "Never married",
        "8": "Living with partner",
        "9": "Unknown",
    },
    "HISPAN_I": {
        "0": "Multiple Hispanic",
        "1": "Puerto Rico",
        "2": "Mexican",
        "3": "Mexican-American",
        "4": "Cuban",
        "5": "Dominican",
        "6": "Central/South American",
        "7": "Other Latin American, n.s.",
        "8": "Other Spanish",
        "9": "Hispanic/Spanish, n.s.",
        "10": "Type refused",
        "11": "Type not ascertained",
        "12": "Not Hispanic/Spanish",
    },
    "RACERPI2": {
        "1": "White only",
        "2": "Black/African American only",
        "3": "American Indian/Alaska Native only",
        "4": "Asian only",
        "5": "Race group not releasable",
        "6": "Multiple race",
    },

    # --- Employment / work status ---
    "DOINGLWA": {
        "1": "Working for pay",
        "2": "With a job, not at work",
        "3": "Looking for work",
        "4": "Unpaid in family business",
        "5": "Not working & not looking",
    },
    "WRKLYR4": {
        "0": "Had job last week",
        "1": "No job last week, had job in last 12m",
        "2": "No job last week, no job in last 12m",
        "3": "Never worked",
    },

    # --- Alcohol / smoking status ---
    "ALCSTAT": {
        "1": "Lifetime abstainer",
        "2": "Former infrequent",
        "3": "Former regular",
        "4": "Current infrequent",
        "5": "Current light",
        "6": "Current moderate",
        "7": "Refused",
        "8": "Not ascertained",
        "9": "Don't know",
        "10": "Unknown status",
    },
    "ALC12MTP": {"1": "Day(s)", "2": "Week(s)", "3": "Month(s)", "4": "Year(s)"},
    "SMKSTAT2": {
        "1": "Current every day smoker",
        "2": "Current some day smoker",
        "3": "Former smoker",
        "4": "Never smoker",
        "5": "Smoker, current status unknown",
        "9": "Unknown if ever smoked",
    },

    # --- Binary health conditions ---
    "AMIGR": {"1": "Yes", "2": "No"},
    "CANEV": {"1": "Yes", "2": "No"},
    "HYPEV": {"1": "Yes", "2": "No"},
    "CHDEV": {"1": "Yes", "2": "No"},
    "AASMEV": {"1": "Yes", "2": "No"},
    "COPDEV": {"1": "Yes", "2": "No"},
    "STREV": {"1": "Yes", "2": "No"},
    "EPHEV": {"1": "Yes", "2": "No"},
    "HRTEV": {"1": "Yes", "2": "No"},

    # --- Ordinal / Likert ---
    "PAIN_2A": {"1": "Never", "2": "Some days", "3": "Most days", "4": "Every day"},
    "PAIN_4": {"1": "A little", "2": "A lot", "3": "Somewhere in between a little and a lot"},
    "ASISAD": {"1": "ALL", "2": "MOST", "3": "SOME", "4": "A LITTLE", "5": "NONE"},
    "ASINERV": {"1": "ALL", "2": "MOST", "3": "SOME", "4": "A LITTLE", "5": "NONE"},
    "ASIWTHLS": {"1": "ALL", "2": "MOST", "3": "SOME", "4": "A LITTLE", "5": "NONE"},
}

# Save codebook
with open(CODEBOOK_PATH, "w") as f:
    json.dump(CODEBOOK, f, indent=2)
print(f"✅ Wrote semantic codebook to {CODEBOOK_PATH}")

# -----------------------------
# 3. Apply mappings in Spark
# -----------------------------
df_semantic = df
dict_rows = []

for col, mapping in CODEBOOK.items():
    if col not in df.columns:
        continue

    # convert numeric → string, join via when()
    mapping_expr = F.when(F.col(col).isNull(), None)
    for k, v in mapping.items():
        mapping_expr = mapping_expr.when(F.col(col) == F.lit(int(k)), F.lit(v))
    mapping_expr = mapping_expr.otherwise(None)

    df_semantic = df_semantic.withColumn(f"{col}_LABEL", mapping_expr)

    for k, v in mapping.items():
        dict_rows.append({"column": col, "code": k, "label": v})

print(f"✅ Added {len(CODEBOOK)} semantic mappings ({len(df_semantic.columns)} total columns).")

# -----------------------------
# 4. Export outputs
# -----------------------------
df_semantic.toPandas().to_csv(OUT_SEMANTIC_CSV, index=False)
pd.DataFrame(dict_rows).to_csv(DICT_CSV, index=False)

with open(SNAPSHOT, "w") as f:
    json.dump(CODEBOOK, f, indent=2)

print(f"📁 Saved outputs:")
print(f"  - Semantic table : {OUT_SEMANTIC_CSV}")
print(f"  - Dictionary     : {DICT_CSV}")
print(f"  - Snapshot JSON  : {SNAPSHOT}")

# -----------------------------
# 5. Review low-cardinality columns not mapped
# -----------------------------
numeric_cols = [c for c, t in df.dtypes if t in ("int", "bigint", "double")]
low_card = []
for c in numeric_cols:
    nunq = df.select(c).distinct().count()
    if 1 < nunq <= 12 and c not in CODEBOOK:
        low_card.append((c, nunq))

if low_card:
    print("\n[Review] Possible coded columns not yet in codebook:")
    for name, nunq in low_card:
        print(f"  - {name}: {nunq} unique values")
else:
    print("\n[Review] All coded columns appear mapped.")

print("\n✅ Iteration 4 Data Integration complete.")

✅ Loaded constructed table: 25,403 rows × 85 columns
✅ Wrote semantic codebook to /content/drive/MyDrive/Infosys722/iteration4_prep_outputs/semantic_codebook_full.json
✅ Added 24 semantic mappings (102 total columns).
📁 Saved outputs:
  - Semantic table : /content/drive/MyDrive/Infosys722/iteration4_prep_outputs/final_integrated_table_semantic.csv
  - Dictionary     : /content/drive/MyDrive/Infosys722/iteration4_prep_outputs/semantic_dictionary.csv
  - Snapshot JSON  : /content/drive/MyDrive/Infosys722/iteration4_prep_outputs/semantic_codebook_snapshot.json

[Review] Possible coded columns not yet in codebook:
  - AHAYFYR: 2 unique values
  - AHCAFYR1: 2 unique values
  - AHCAFYR3: 2 unique values
  - AHCNOYR2: 9 unique values
  - AHERNOY2: 9 unique values
  - ANX_1: 5 unique values
  - ANX_2: 2 unique values
  - ANX_3R: 3 unique values
  - ARX12_2: 2 unique values
  - ARX12_3: 2 unique values
  - ASIEFFRT: 5 unique values
  - ASIHIVT: 2 unique values
  - ASIHOPLS: 5 unique values
  - 

In [ ]:
# =========================================
# 3.5 Reformatting (Iteration 4)
# Prepare pre-transformation base tables
# =========================================

import numpy as np


# Input from Step 3.4
IN_PATH = OUT_DIR / "final_integrated_table_semantic.csv"
assert IN_PATH.exists(), f"❌ Input not found: {IN_PATH}"

# -----------------------------
# 1) Load Data
# -----------------------------
df_sem = spark.read.option("header", True).csv(str(IN_PATH), inferSchema=True)
print(f"✅ Loaded semantic table: {df_sem.count():,} rows × {len(df_sem.columns)} columns")

# Convert to pandas for flexible reformatting
pdf_sem = df_sem.toPandas()

# -----------------------------
# 2) Column partitions
# -----------------------------
label_cols = [c for c in pdf_sem.columns if c.endswith("_LABEL")]
core_cols  = [c for c in pdf_sem.columns if c not in label_cols]

RID_COL = "RID"
TARGET_COL = "AMIGR"

def ordered(cols):
    """Ensure RID and target appear first."""
    pref = [x for x in [RID_COL, TARGET_COL] if x in cols]
    rest = [c for c in cols if c not in pref]
    return pref + rest

# Reporting: keep *_LABEL only
reporting_cols = [RID_COL, TARGET_COL] + label_cols
reporting_cols = [c for c in reporting_cols if c in pdf_sem.columns]
df_reporting = pdf_sem[ordered(reporting_cols)].copy()

# Modeling base: drop *_LABEL
df_modeling_base = pdf_sem[ordered(core_cols)].copy()

# -----------------------------
# 3) DType harmonization
# -----------------------------
# All *_LABEL columns → string
for c in label_cols:
    if c in df_reporting.columns:
        df_reporting[c] = df_reporting[c].astype("string")

# Enforce RID and target as integers
for frame in (df_reporting, df_modeling_base):
    if RID_COL in frame.columns:
        frame[RID_COL] = pd.to_numeric(frame[RID_COL], errors="coerce").astype("Int64")
    if TARGET_COL in frame.columns:
        frame[TARGET_COL] = pd.to_numeric(frame[TARGET_COL], errors="coerce").astype("Int64")

# Integer-count re-clamp safety
int_count_ranges = {
    "ALC12MYR": (0, 365),
    "BEDDAYR": (0, 365),
    "YRSWRKPA": (0, 35),
}
for col, (lo, hi) in int_count_ranges.items():
    if col in df_modeling_base.columns:
        s = pd.to_numeric(df_modeling_base[col], errors="coerce")
        s = np.rint(s).astype("Int64").clip(lower=lo, upper=hi)
        df_modeling_base[col] = s

# -----------------------------
# 4) Quality checks
# -----------------------------
def qa_print(frame, name):
    n_rows, n_cols = frame.shape
    n_missing_rows = frame.isna().any(axis=1).sum()
    print(f"[QA] {name}: {n_rows}×{n_cols} | rows-with-NA={n_missing_rows}")

qa_print(df_reporting, "reporting")
qa_print(df_modeling_base, "modeling_base")

# Check duplicate RID
for frame, name in ((df_reporting, "reporting"), (df_modeling_base, "modeling_base")):
    if RID_COL in frame.columns:
        dup = frame[RID_COL].duplicated(keep=False).sum()
        print(f"[QA] {name}: duplicated RID count = {dup}")

# Alignment sanity
if RID_COL in df_reporting.columns and RID_COL in df_modeling_base.columns:
    aligned = df_reporting[RID_COL].fillna(-1).astype("Int64").equals(
        df_modeling_base[RID_COL].fillna(-1).astype("Int64")
    )
    print(f"[QA] RID alignment between reporting and modeling_base: {aligned}")

# -----------------------------
# 5) Save Outputs (CSV + Parquet)
# -----------------------------
REPORT_CSV = OUT_DIR / "final_reporting_table.csv"
MODEL_CSV  = OUT_DIR / "final_modeling_table_reformatted.csv"
REPORT_PQ  = OUT_DIR / "final_reporting_table.parquet"
MODEL_PQ   = OUT_DIR / "final_modeling_table_reformatted.parquet"

df_reporting.to_csv(REPORT_CSV, index=False)
df_modeling_base.to_csv(MODEL_CSV, index=False)

try:
    df_reporting.to_parquet(REPORT_PQ, index=False)
    df_modeling_base.to_parquet(MODEL_PQ, index=False)
    print(f"[OK] Saved CSV + Parquet versions:")
    print(f"     - {REPORT_CSV.name} / {REPORT_PQ.name}")
    print(f"     - {MODEL_CSV.name}  / {MODEL_PQ.name}")
except Exception as e:
    print("[Warn] Parquet save failed; CSVs only.")
    print("       Error:", e)

print("✅ Step 3.5 Reformatting complete — ready for Step 4 Transformation.")


✅ Loaded semantic table: 25,403 rows × 102 columns
[QA] reporting: 25403×20 | rows-with-NA=0
[QA] modeling_base: 25403×84 | rows-with-NA=0
[QA] reporting: duplicated RID count = 0
[QA] modeling_base: duplicated RID count = 0
[QA] RID alignment between reporting and modeling_base: True
[OK] Saved CSV + Parquet versions:
     - final_reporting_table.csv / final_reporting_table.parquet
     - final_modeling_table_reformatted.csv  / final_modeling_table_reformatted.parquet
✅ Step 3.5 Reformatting complete — ready for Step 4 Transformation.
